# Spatial Domain Design Workflow

This notebook demonstrates the workflow for spatial domain design, including image processing, model training, and evaluation.

## Table of Contents
1. [Imports](#imports)
2. [Image Region Extraction](#image-region-extraction)
3. [Batch Processing](#batch-processing)
4. [Model Definition](#model-definition)
5. [Dataset Preparation](#dataset-preparation)
6. [Training and Evaluation](#training-and-evaluation)
7. [Visualization](#visualization)


## Imports

Import required libraries for image processing, deep learning, and visualization.


In [ ]:
import os
import numpy as np
from PIL import Image
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import matplotlib.pyplot as plt
import torch.optim as optim
from sklearn.metrics import confusion_matrix
import scipy.io as sio

# Disable CUDA optimizations for reproducibility
torch.backends.cudnn.enabled = False


## Image Region Extraction

Extract four regions from PNG images and compute differences between specific regions.


In [ ]:
def extract_four_regions(image_path, output_path=None, regions=None):
    """
    Extract four regions from a PNG image and save as a (4, 300, 300) matrix.
    
    Args:
        image_path: Path to the PNG image file
        output_path: Output file path (optional, saves as .npy if provided)
        regions: List of four region coordinates in format [(x1, y1, x2, y2), ...]
                 If None, uses default corner regions
    
    Returns:
        Result array with shape (2, 300, 300) containing region differences
    """
    try:
        img = Image.open(image_path)
    except Exception as e:
        print(f"Failed to open image: {e}")
        return None
    
    img_array = np.array(img)
    
    if regions is None:
        regions = [
            (132, 3633, 1632, 5120),
            (3630, 3558, 5120, 5058),
            (36, 101, 1536, 1601),
            (3577, 30, 5077, 1530)
        ]
    
    if len(regions) != 4:
        print("Four regions are required")
        return None
    
    extracted_regions = []
    for i, (x1, y1, x2, y2) in enumerate(regions):
        region = img_array[x1:x2, y1:y2]
        h, w = region.shape[:2]
        if h != w:
            max_dim = max(h, w)
            pad_h = max_dim - h
            pad_w = max_dim - w
            if region.ndim == 3:
                region = np.pad(region, ((0, pad_h), (0, pad_w), (0, 0)), mode="constant", constant_values=0)
            else:
                region = np.pad(region, ((0, pad_h), (0, pad_w)), mode="constant", constant_values=0)
        
        if region.shape[0] != 300 or region.shape[1] != 300:
            region_img = Image.fromarray(region)
            region_img = region_img.resize((300, 300))
            region = np.array(region_img)
        
        extracted_regions.append(region)
    
    if len(extracted_regions) == 4:
        array = np.stack(extracted_regions, axis=0).astype(np.int16)
        diff1 = array[0] - array[2]
        diff2 = array[1] - array[3]
        result_array = np.stack([diff1, diff2], axis=0)
        
        if output_path:
            np.save(output_path, result_array)
            print(f"Saved to: {output_path}")
        
        return result_array
    else:
        print("Failed to extract four valid regions")
        return None


## Batch Processing

Process multiple PNG images in a directory and save extracted regions as numpy arrays.


In [ ]:
# Configuration - Update these paths according to your environment
input_folder = "./dataset/train/data"
output_folder = "./dataset/train/data/out"

# Create output folder if it doesn't exist
if not os.path.exists(output_folder):
    os.makedirs(output_folder)
    print(f"Created output folder: {output_folder}")

# Optional: Custom region coordinates
custom_regions = None

# Get all PNG files in input folder
png_files = [f for f in os.listdir(input_folder) if f.lower().endswith(".png")]
png_files.sort()

print(f"Found {len(png_files)} PNG files")

# Process each PNG file
for i, png_file in enumerate(png_files):
    image_path = os.path.join(input_folder, png_file)
    original_name = os.path.splitext(png_file)[0]
    output_filename = f"{original_name}.npy"
    output_path = os.path.join(output_folder, output_filename)
    
    print(f"Processing file {i+1}/{len(png_files)}: {png_file}")
    result = extract_four_regions(image_path, output_path, custom_regions)
    
    if result is not None:
        print(f"Successfully extracted 2 regions, shape: {result.shape}")
    else:
        print(f"Extraction failed: {png_file}")
    
    print("-" * 50)

print("Batch processing completed!")


In [ ]:
# Configuration - Update these paths according to your environment
input_folder = "./dataset/test/data"
output_folder = "./dataset/test/data/out"

# Create output folder if it doesn't exist
if not os.path.exists(output_folder):
    os.makedirs(output_folder)
    print(f"Created output folder: {output_folder}")

# Optional: Custom region coordinates
custom_regions = None

# Get all PNG files in input folder
png_files = [f for f in os.listdir(input_folder) if f.lower().endswith(".png")]
png_files.sort()

print(f"Found {len(png_files)} PNG files")

# Process each PNG file
for i, png_file in enumerate(png_files):
    image_path = os.path.join(input_folder, png_file)
    original_name = os.path.splitext(png_file)[0]
    output_filename = f"{original_name}.npy"
    output_path = os.path.join(output_folder, output_filename)
    
    print(f"Processing file {i+1}/{len(png_files)}: {png_file}")
    result = extract_four_regions(image_path, output_path, custom_regions)
    
    if result is not None:
        print(f"Successfully extracted 2 regions, shape: {result.shape}")
    else:
        print(f"Extraction failed: {png_file}")
    
    print("-" * 50)

print("Batch processing completed!")


## Model Definition

Define the neural network model using depthwise separable convolutions.


In [61]:
# Depthwise Separable Convolution Blocks

class DepthwiseSeparableConv(nn.Module):
    """MobileNet V1 style depthwise separable convolution block"""
    def __init__(self, in_channels, out_channels, kernel_size=3, stride=1, padding=1):
        super().__init__()
        # Depthwise convolution
        self.depthwise = nn.Conv2d(
            in_channels, in_channels,
            kernel_size=kernel_size,
            stride=stride,
            padding=padding,
            groups=in_channels,
            bias=False
        )
        # Pointwise convolution
        self.pointwise = nn.Conv2d(
            in_channels, out_channels,
            kernel_size=1,
            stride=1,
            padding=0,
            bias=False
        )
        self.bn1 = nn.BatchNorm2d(in_channels)
        self.bn2 = nn.BatchNorm2d(out_channels)

    def forward(self, x):
        x = F.leaky_relu(self.bn1(self.depthwise(x)))  # Depthwise + BN + ReLU
        x = F.leaky_relu(self.bn2(self.pointwise(x)))  # Pointwise + BN + ReLU
        return x


class DoubleConv2dBN(nn.Module):
    """Double depthwise separable convolution block"""
    def __init__(self, in_channels, out_channels, kernel_size=3, stride=1, padding=1):
        super().__init__()
        self.conv1 = DepthwiseSeparableConv(
            in_channels, out_channels,
            kernel_size=kernel_size,
            stride=stride,
            padding=padding
        )
        self.conv2 = DepthwiseSeparableConv(
            out_channels, out_channels,
            kernel_size=kernel_size,
            stride=stride,
            padding=padding
        )

    def forward(self, x):
        x = self.conv1(x)
        x = self.conv2(x)
        return x


class Deconv2dBN(nn.Module):
    """Transposed convolution block"""
    def __init__(self, in_channels, out_channels, kernel_size=2, stride=2):
        super().__init__()
        self.conv1 = nn.ConvTranspose2d(
            in_channels, out_channels,
            kernel_size=kernel_size,
            stride=stride, bias=True
        )
        self.bn1 = nn.BatchNorm2d(out_channels)

    def forward(self, x):
        return F.leaky_relu(self.bn1(self.conv1(x)))


# U-Net Models

class NewUnet(nn.Module):
    """Lightweight U-Net architecture using depthwise separable convolutions (simplified)"""
    def __init__(self):
        super(NewUnet, self).__init__()
        self.pointwise = nn.Conv2d(2, 12, kernel_size=1, stride=1, padding=0, bias=False)
        self.layer2_conv = DoubleConv2dBN(12, 24)
        self.layer3_conv = DoubleConv2dBN(24, 12)
        self.layer4_conv = nn.Conv2d(12, 3, kernel_size=3, stride=1, padding=1, bias=True)
        self.deconv1 = Deconv2dBN(24, 12)
        self.sigmoid = nn.Sigmoid()
        
    def forward(self, x):
        # 1. Simulate optoelectronic hybrid neural network: uncomment the following two lines
        # conv1, PSF, aft_conv = self.mycustomlayer(x)
        # conv1_2 = self.pointwise(conv1)
        # 2. Simulate pure electrical neural network: uncomment the following line
        # conv1_2 = self.layer1_conv(x)
        # 3. Validate with experimental data: use the following line (pointwise)
        conv1_2 = self.pointwise(x)
        pool1 = F.max_pool2d(conv1_2, 2)
        
        conv2 = self.layer2_conv(pool1)
        convt1 = self.deconv1(conv2)
        concat1 = torch.cat([convt1, conv1_2], dim=1)
        conv3 = self.layer3_conv(concat1)
        
        conv4 = self.layer4_conv(conv3)
        outp = self.sigmoid(conv4)
        return outp


class MetaUnet2(nn.Module):
    """Lightweight U-Net architecture using depthwise separable convolutions"""
    def __init__(self):
        super(MetaUnet2, self).__init__()
        self.pointwise = nn.Conv2d(2, 12, kernel_size=1, stride=1, padding=0, bias=False)
        self.layer2_conv = DoubleConv2dBN(12, 24)
        self.layer3_conv = DoubleConv2dBN(24, 48)
        self.deconv1 = Deconv2dBN(48, 24)
        self.layer4_conv = DoubleConv2dBN(48, 24)
        self.deconv2 = Deconv2dBN(24, 12)
        self.layer5_conv = DoubleConv2dBN(24, 12)
        self.layer6_conv = nn.Conv2d(12, 3, kernel_size=3, stride=1, padding=1, bias=True)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        # 1. Simulate optoelectronic hybrid neural network: uncomment the following two lines
        # conv1, PSF, aft_conv = self.mycustomlayer(x)
        # conv1_2 = self.pointwise(conv1)
        # 2. Simulate pure electrical neural network: uncomment the following line
        # conv1_2 = self.layer1_conv(x)
        # 3. Validate with experimental data: use the following line (pointwise)
        conv1 = self.pointwise(x)
        pool1 = F.max_pool2d(conv1, 2)
        conv2 = self.layer2_conv(pool1)
        pool2 = F.max_pool2d(conv2, 2)
        conv3 = self.layer3_conv(pool2)
        convt1 = self.deconv1(conv3)
        concat1 = torch.cat([convt1, conv2], dim=1)
        conv4 = self.layer4_conv(concat1)
        convt2 = self.deconv2(conv4)
        concat2 = torch.cat([convt2, conv1], dim=1)
        conv5 = self.layer5_conv(concat2)
        conv6 = self.layer6_conv(conv5)
        outp = self.sigmoid(conv6)
        return outp


# Test models with random input
print("Testing NewUnet...")
model1 = NewUnet()
inp = torch.rand(10, 2, 300, 300)
outp1 = model1(inp)
print(f"NewUnet output shape: {outp1.shape}")

print("Testing MetaUnet2...")
model2 = MetaUnet2()
outp2 = model2(inp)
print(f"MetaUnet2 output shape: {outp2.shape}")

print("All models initialized successfully!")


Testing NewUnet...
NewUnet output shape: torch.Size([10, 3, 300, 300])
Testing MetaUnet2...
MetaUnet2 output shape: torch.Size([10, 3, 300, 300])
All models initialized successfully!


## Dataset Preparation

Define custom dataset class and prepare training/test data loaders.


In [ ]:
class CustomDataset(Dataset):
    """Custom dataset for loading extracted regions and labels"""
    def __init__(self, data_folder, label_folder, transform=None):
        self.data_folder = data_folder
        self.label_folder = label_folder
        self.transform = transform
        self.data_files = [f for f in os.listdir(data_folder) if f.endswith(".npy")]
        self.data_files.sort()
        
        self.valid_files = []
        for f in self.data_files:
            label_name = f
            if os.path.exists(os.path.join(label_folder, label_name)):
                self.valid_files.append(f)
        
        print(f"Found {len(self.valid_files)} valid data-label pairs")

    def __len__(self):
        return len(self.valid_files)

    def __getitem__(self, idx):
        data_name = self.valid_files[idx]
        label_name = data_name  # Use same filename as data
        
        data_path = os.path.join(self.data_folder, data_name)
        label_path = os.path.join(self.label_folder, label_name)
        
        data = np.load(data_path).astype(np.float32)
        label = np.load(label_path).astype(np.int64)
        
        # Resize label from 1500x1500 to 300x300 to match input size
        label_tensor = torch.from_numpy(label)
        if label_tensor.shape != (300, 300):
            # F.interpolate requires float type, convert back to long after
            label_tensor = F.interpolate(label_tensor.unsqueeze(0).unsqueeze(0).float(), 
                                        size=(300, 300), mode='nearest').squeeze(0).squeeze(0).long()
        
        return torch.from_numpy(data), label_tensor

# Dataset paths - Update according to your environment
train_data_folder = "./dataset/train/data/out"
train_label_folder = "./dataset/train/labels"
test_data_folder = "./dataset/test/data/out"
test_label_folder = "./dataset/test/labels"

# Create datasets
train_dataset = CustomDataset(train_data_folder, train_label_folder)
test_dataset = CustomDataset(test_data_folder, test_label_folder)

# Create data loaders
batch_size = 4
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print(f"Training samples: {len(train_dataset)}")
print(f"Testing samples: {len(test_dataset)}")


## Training and Evaluation

Define training loop with learning rate scheduling and mIoU evaluation.


In [ ]:
def calculate_miou(predictions, labels, num_classes):
    """
    Calculate Mean Intersection over Union (mIoU) metric.
    
    Args:
        predictions: Predicted segmentation maps
        labels: Ground truth segmentation maps
        num_classes: Number of classes
    
    Returns:
        Mean IoU across all classes
    """
    if len(predictions.shape) == 2:
        predictions = np.expand_dims(predictions, axis=0)
        labels = np.expand_dims(labels, axis=0)
    
    miou_list = []
    for i in range(predictions.shape[0]):
        pred = predictions[i]
        label = labels[i]
        confusion = confusion_matrix(label.flatten(), pred.flatten(), labels=range(num_classes))
        TP = np.diag(confusion)
        FP = np.sum(confusion, axis=0) - TP
        FN = np.sum(confusion, axis=1) - TP
        union = TP + FP + FN
        iou = np.divide(TP, union, out=np.full_like(TP, np.nan, dtype=float), where=union != 0)
        miou = np.nanmean(iou)
        miou_list.append(miou)
    
    return np.mean(miou_list)

def get_segmentation_results(outputs):
    """Convert model outputs to segmentation masks"""
    return torch.argmax(outputs, dim=1)


In [ ]:
# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Initialize model
model = MetaUnet2().to(device)

# Define loss function with class weights
class_weights = torch.tensor([1.0, 2.0, 2.0], dtype=torch.float32).to(device)
criterion = nn.CrossEntropyLoss(weight=class_weights)

# Define optimizer
learning_rate = 1e-3
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

# Track learning rates
learning_rates = [learning_rate]


In [ ]:
def train_model(model, criterion, optimizer, num_epochs=40):
    """
    Train the model with learning rate decay and validation.
    """
    initial_lr = optimizer.param_groups[0]["lr"]
    best_val_miou = 0.0
    
    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0
        
        for i, data in enumerate(train_loader, 0):
            inputs, labels = data
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
            
            if i % 20 == 19:
                print(f"Epoch {epoch+1}, Batch {i+1}, Loss: {running_loss / 20:.4f}")
                running_loss = 0.0
        
        if (epoch + 1) % 20 == 0:
            for param_group in optimizer.param_groups:
                param_group["lr"] = initial_lr
            print(f"Warm restart activated! Learning rate reset to: {initial_lr:.8f}")
        else:
            for param_group in optimizer.param_groups:
                param_group["lr"] *= 0.9
        
        current_lr = optimizer.param_groups[0]["lr"]
        learning_rates.append(current_lr)
        print(f"Epoch {epoch+1}, Learning Rate: {current_lr:.8f}")
        print(f"Finished Epoch {epoch+1}/{num_epochs}")
        
        model.eval()
        val_loss = 0.0
        miou_list = []
        
        with torch.no_grad():
            for data in test_loader:
                inputs, labels = data
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                predictions = get_segmentation_results(outputs)
                predictions = predictions.cpu().numpy()
                labels_np = labels.cpu().numpy()
                miou = calculate_miou(predictions, labels_np, num_classes=3)
                miou_list.append(miou)
                val_loss += loss.item()
        
        mean_miou = np.mean(miou_list)
        print(f"Validation Loss: {val_loss / len(test_loader):.4f}")
        print(f"Validation mIoU: {mean_miou:.4f}")
        
        if mean_miou > best_val_miou:
            best_val_miou = mean_miou
            torch.save(model.state_dict(), "best_model.pth")
            print(f"New best model saved with mIoU: {best_val_miou:.4f}")
        
        print("-" * 50)

# Start training
train_model(model, criterion, optimizer, num_epochs=40)


## Visualization

Visualize training results and learning rate curve.


In [ ]:
# Plot learning rate decay curve
plt.figure(figsize=(10, 5))
plt.plot(learning_rates)
plt.xlabel("Epoch")
plt.ylabel("Learning Rate")
plt.title("Learning Rate Decay")
plt.grid(True)
plt.show()
